In [1]:
import os

import pickle

import numpy as np
import pandas as pd
from sklearn.utils import Bunch

from efaar_benchmarking.efaar import tvn_on_controls
from efaar_benchmarking.benchmarking import known_relationship_benchmark, compound_gene_benchmark


dataset = "rxrx3"
save_results = False

# Load Data

In [ ]:
def load_rxrx3(data_path: str = "data/rxrx3/"):
    """Load Recursion's rxrx3 dataset, updated with unblinded data metadata.

    Note that you have to download the RPIE CNNBC embeddings and metadata manually from
    https://rxrx3.rxrx.ai/downloads as a sign-in process is required.
    Data is expected to be present as unzipped files in the `data_path` directory passed as an argument here.

    Args:
        data_path (str): Path to the data directory. Default is "data/rxrx3/".

    Returns:
        tuple: A tuple containing two pandas DataFrames:
            - features: DataFrame containing the extracted features.
            - metadata: DataFrame containing the metadata information.

    """
    metadata = pd.read_csv(os.path.join(data_path, "metadata_rxrx3_public.csv"))

    metadata["perturbation"] = metadata["treatment"].apply(lambda x: x.split("_")[0] if "_control" not in x else x)
    print("Metadata shape:", metadata.shape)

    embeddings = pd.read_parquet(os.path.join(data_path, "embeddings"))
    print("Embeddings shape:", embeddings.shape)

    rxrx3 = metadata.merge(embeddings, on="well_id")
    feature_cols = [x for x in list(rxrx3.columns) if x.startswith("feature_")]
    rxrx3 = rxrx3.dropna(subset=feature_cols)
    rxrx3.reset_index(drop=True, inplace=True)
    print("Final shape:", rxrx3.shape)

    return (rxrx3[feature_cols], rxrx3[[x for x in list(rxrx3.columns) if x not in feature_cols]])


features, metadata = load_rxrx3("<your_path_here>")

# Compute recall on biological relationships

In [ ]:
pert_colname = "perturbation"
experiment_colname = "experiment_name"
control_key = "EMPTY_control"

print("Computing TVN embedding with EEFAR...")
embeddings_tvn = tvn_on_controls(
    features.astype(float).values, metadata[[pert_colname, experiment_colname]], pert_col=pert_colname, control_key=control_key, batch_col=experiment_colname)
embeddings_tvn = pd.DataFrame(embeddings_tvn, index=metadata.index, columns=[
                              f"feature_{i}" for i in range(embeddings_tvn.shape[1])])
                              
print("Aggregating perturbations...")

merged = pd.concat([metadata, embeddings_tvn], axis=1)
merged = merged[~((merged['perturbation_type'] == 'COMPOUND') & (merged['perturbation'].str.contains('_control')))]
agg_func = {col: 'mean' for col in merged.columns if col.startswith('feature_')}
map_data = merged.groupby(
    ['perturbation_type', 'perturbation', 'concentration'], dropna=False
).agg(agg_func).reset_index()

features_cols = [col for col in map_data.columns if col.startswith('feature_')]
metadata_cols = [col for col in map_data.columns if col not in features_cols]

In [ ]:
pert_signal_pval_cutoff = 0.05
recall_thr_pairs = [(0.05, 0.95)]

print("Computing recall...")
metrics = known_relationship_benchmark(Bunch(
    metadata=map_data[metadata_cols], features=map_data[features_cols]), recall_thr_pairs=recall_thr_pairs, pert_col=pert_colname)
print("Recall Results", metrics[list(metrics.columns)[::-1]])

if save_results:
    with open(f'data/{dataset}_map_cache.pkl', 'wb') as f:
        pickle.dump(map_data, f)  
    with open(f'data/{dataset}_metadata.pkl', 'wb') as f:
        pickle.dump(metadata, f) 

print("Done!")

# Compute mAP on compound-gene relationships

In [ ]:
print("Computing mAP...")
map_score, curves = compound_gene_benchmark(
    map_data, nM_activity_threshold=1000, pert_col=pert_colname,
)
print("mAP Results", map_score)

if save_results:
    with open(f'data/{dataset}_map_score.pkl', 'wb') as f:
        pickle.dump(map_score, f)  
    with open(f'data/{dataset}_curves.pkl', 'wb') as f:
        pickle.dump(curves, f)

print("Done!")